# 🎬 4K AI Video Upscaling & Enhancement in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arkyu2077/ai-video-upscaler/blob/main/examples/4K_AI_Video_Upscaling_Colab_Tutorial.ipynb)

Welcome to the official interactive Google Colab tutorial for **[VideoUpscaler](https://videoupscaler.co)** — the cloud-native AI video super-resolution and restoration platform. With VideoUpscaler, you can upscale low-resolution 480p/720p videos into crisp **1080p, 2K, or 4K Ultra HD** without needing a local GPU.

### 🌐 Core Capabilities & Platform Resources
- **[AI Video Upscaler Platform](https://videoupscaler.co)**: Upload and enhance videos directly in your browser or via our developer API.
- **[4K Video Upscaler](https://videoupscaler.co/4k-video-upscaler)**: Multi-frame temporal AI super-resolution up to 3840×2160 (4K UHD) with crisp edge, face, and text reconstruction.
- **[Batch Video Upscaler](https://videoupscaler.co/batch-video-upscaler)**: Process entire folders, media archives, and multi-clip production pipelines concurrently.
- **[AI Video Denoising](https://videoupscaler.co/denoise-video)**: Remove sensor grain, low-light ISO noise, and MPEG compression block artifacts while preserving fine detail.
- **[AI Frame Interpolation (60fps)](https://videoupscaler.co/frame-interpolation)**: Convert choppy 24fps or 30fps footage into silky-smooth 60fps motion.
- **[AI Video Stabilization](https://videoupscaler.co/stabilize-video)**: Eliminate camera shake and handheld jitter from action, drone, and mobile clips.
- **[Workflow Integrations (n8n, MCP, Make, Zapier)](https://videoupscaler.co/integrations)**: Connect VideoUpscaler to Claude Desktop, Cursor, n8n, and automated media pipelines.
- **[Developer API & Workflow Docs](https://videoupscaler.co/docs/workflows)**: Full REST API specification, webhook/polling patterns, and production recipes.


## 1. Install the `videoupscaler` Python SDK & CLI

First, let's install the official zero-dependency Python SDK and CLI directly from GitHub, along with `requests` for raw REST API examples.

In [ ]:
!pip install -q git+https://github.com/arkyu2077/videoupscaler-cli.git requests

## 2. Configure Your API Key

Get your API key from your [VideoUpscaler Dashboard](https://videoupscaler.co) (`Settings -> API Keys`) and set `VIDEOUPSCALER_API_KEY` below. You can also inspect your credit balance and per-second pricing across `1080p`, `2k`, and `4k`.

In [ ]:
import os
from getpass import getpass
from videoupscaler import Client

if not os.environ.get("VIDEOUPSCALER_API_KEY"):
    os.environ["VIDEOUPSCALER_API_KEY"] = getpass("Enter your VIDEOUPSCALER_API_KEY (vu_live_...): ")

client = Client()
account = client.get_account()
print(f"Available Credits: {account['credits']}")
print(f"Credits per second: {account['credits_per_second']}")

## 3. Submit a 4K 60fps Video Upscale Task (Python SDK & REST API)

Whether you are enhancing vintage footage with the [4K Video Upscaler](https://videoupscaler.co/4k-video-upscaler), cleaning up noisy clips with [AI Video Denoising](https://videoupscaler.co/denoise-video), smoothing motion via [AI Frame Interpolation](https://videoupscaler.co/frame-interpolation), or fixing shaky camera work with [AI Video Stabilization](https://videoupscaler.co/stabilize-video), you can submit jobs in a single API call.

Every request supports an `Idempotency-Key` header so network retries never double-charge your account.

In [ ]:
import uuid
import requests

API_BASE = "https://videoupscaler.co/api/v1"
HEADERS = {
    "Authorization": f"Bearer {os.environ['VIDEOUPSCALER_API_KEY']}",
    "Content-Type": "application/json",
}

sample_video_url = "https://example.com/sample-720p-clip.mp4"

# Option A: Using the Python SDK
task = client.upscale(
    video_url=sample_video_url,
    resolution="4k",
)
print(f"Created 4K Task via SDK: {task['id']} (status={task['status']})")

# Option B: Direct REST API call for 4K 60fps upscaling & enhancement
response = requests.post(
    f"{API_BASE}/upscale",
    headers={**HEADERS, "Idempotency-Key": str(uuid.uuid4())},
    json={
        "video_url": sample_video_url,
        "resolution": "4k",
        "fps": 60,
        "denoise": True,
        "stabilize": True,
    },
    timeout=300,
)
response.raise_for_status()
rest_task = response.json()
print(f"Created 4K 60fps Task via REST API: {rest_task['id']}")

## 4. Poll Task Completion with Long-Polling (`?wait=45`)

Instead of hammering the endpoint every few seconds, you can pass `?wait=45` to `GET /api/v1/tasks/{id}?wait=45` so the server holds the connection open for up to 45 seconds and returns immediately as soon as the task state changes or completes. See the [Developer Workflow Docs](https://videoupscaler.co/docs/workflows) for more details on long-polling and SSE streaming.

In [ ]:
def wait_for_task_long_poll(task_id: str, wait_seconds: int = 45) -> dict:
    """Poll a VideoUpscaler task using server-side long-polling (?wait=45)."""
    while True:
        res = requests.get(
            f"{API_BASE}/tasks/{task_id}?wait={wait_seconds}",
            headers=HEADERS,
            timeout=wait_seconds + 15,
        )
        res.raise_for_status()
        current = res.json()
        print(f"Task {current['id']} -> status: {current['status']}")
        if current["status"] in ("completed", "failed"):
            return current

completed_task = wait_for_task_long_poll(rest_task["id"], wait_seconds=45)
if completed_task["status"] == "completed":
    print("✅ 4K Output URL:", completed_task["output_url"])
else:
    print("❌ Task failed:", completed_task.get("error"))

## 5. Batch-Process a Folder of Videos

Need to upscale an entire directory of local `.mp4`, `.mov`, or `.webm` files in Colab or Google Drive? Combine our signed upload flow (`client.upload_file`) with concurrent long-polling, or explore the web-based [Batch Video Upscaler](https://videoupscaler.co/batch-video-upscaler) and [Workflow Integrations](https://videoupscaler.co/integrations) for zero-code automation.

In [ ]:
import glob
from pathlib import Path

def batch_upscale_folder(input_dir: str, output_dir: str, resolution: str = "4k"):
    """Upload and upscale all videos in a folder to 4K."""
    os.makedirs(output_dir, exist_ok=True)
    video_files = sorted(
        p for ext in ("*.mp4", "*.mov", "*.webm")
        for p in glob.glob(os.path.join(input_dir, ext))
    )
    if not video_files:
        print(f"No video files found in {input_dir}")
        return []

    submitted = []
    for file_path in video_files:
        print(f"Uploading {file_path}...")
        upload = client.upload_file(file_path)
        t = client.upscale(upload["video_url"], resolution=resolution)
        submitted.append((file_path, t["id"]))
        print(f"  -> Submitted task {t['id']} for {Path(file_path).name}")

    results = []
    for file_path, task_id in submitted:
        final = wait_for_task_long_poll(task_id, wait_seconds=45)
        if final["status"] == "completed":
            out_path = os.path.join(output_dir, f"{Path(file_path).stem}_{resolution}.mp4")
            client.download(final["output_url"], out_path)
            print(f"  -> Saved {out_path}")
            results.append(out_path)
    return results

# Example usage on a local Colab directory:
# batch_upscale_folder("/content/input_videos", "/content/output_4k_videos", resolution="4k")